In [2]:
import xwrf
import glob
import pickle

import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import numpy as np
from wrf import latlon_coords, ll_to_xy, xy_to_ll
from netCDF4 import Dataset
from wrf import getvar #for RH

Extract urban morphological parameters at the location of each station - Table 3

In [3]:
#identify URBAN-ONLY stations
# Define file path and domain boundaries
file_luindex = "/g/data/fy29/mf9078/run_WRF/etamodified/output/4out/animation/wrfout_d02_2017-01-12_21:00:00"
# Load the geo_em file
#ds = xr.open_dataset(file_luindex)

# mask
lat_min, lat_max = -34.14, -33.55
lon_min, lon_max = 150.57, 151.37

#14 DCCEEW stations in Sydney
slat = [-33.93175,-33.86433,-33.78113,-33.93132,-33.91766,-33.89156,-33.91619,-33.61641,-34.30621,-33.79512,-34.05170,-33.79424,-34.05610,-34.04168]
slon = [151.24278,151.16395,151.15090,150.90727,150.76192,151.04610,151.13577,150.74731,150.58061,150.76677,150.49819,150.91417,150.81220,150.69013]
snm = ['RK','RE','LD','LL','BY','CA','ED','RD','BO','SS','OE','PT','CN','CD']
stations = {'RK':'Randwick','RE':'Rozelle','LD':'Lindfield','LL':'Liverpool','BY':'Bringelly','CA':'Chullora','ED':'Earlwood','RD':'Richmond','BO':'Bargo','SS':'St Marys','OE':'Oakdale','PT':'Prospect','CN':'Campbelltown','CD':'Camden'}

snm_indomain=[]
slat_indomain=[]
slon_indomain=[]
# Add station labels
for i, (lon, lat, name) in enumerate(zip(slon, slat, snm)):
    if (lat >= lat_min) & (lat <= lat_max) & (lon >= lon_min) & (lon <= lon_max):
        snm_indomain.append(name)
        slat_indomain.append(lat)
        slon_indomain.append(lon)
    #else:
        #print(f"Station {name} falls outside the domain")

#xy of WRF grids corresponding to the stations locations within the latlon limit
x_y = ll_to_xy(Dataset(file_luindex), slat_indomain, slon_indomain, stagger=None, as_int=True)
sn = x_y[1].values
we = x_y[0].values

#lat/lon of WRF grids corresponding to the stations locations
l_l = xy_to_ll(Dataset(file_luindex), we, sn, stagger=None)
slat_wrfgrids=l_l[0].values
slon_wrfgrids=l_l[1].values

#xy and ll of WRF grids corresponding to the stations locations - ONLY URBAN
ds = xr.open_dataset(file_luindex)

snm_urban=[]
sn_urban=[]
we_urban=[]
slat_urban=[]
slon_urban=[]

for i in range(len(snm_indomain)):
    if ds['LU_INDEX'].isel(south_north=sn[i], west_east=we[i]).item() > 50:
        snm_urban.append(snm_indomain[i])
        sn_urban.append(sn[i])
        we_urban.append(we[i])      
             
#lat/lon of WRF grids corresponding to the stations locations - URBAN ONLY
l_l_urban = xy_to_ll(Dataset(file_luindex), we_urban, sn_urban, stagger=None)
slat_wrfurban=l_l_urban[0].values
slon_wrfurban=l_l_urban[1].values


In [ ]:
# Extract the parameters
file = "/g/data/fy29/mf9078/run_WRF/etamodified/setup/4rad/wrfinput_d02"
parameters = ["FRC_URB2D", "BUILD_AREA_FRACTION", "BUILD_HEIGHT", "BUILD_SURF_RATIO", "HGT", "LU_INDEX"]

# Open dataset
ds = xr.open_dataset(file)

# Extract data for all stations
data = []
for station, sn, we in zip(snm_urban, sn_urban, we_urban):
    station_data = {'Station': station, 'south_north': sn, 'west_east': we}
    
    for param in parameters:
        value = ds[param].isel(Time=0, south_north=sn, west_east=we).values
        station_data[param] = value
    
    data.append(station_data)

# Create DataFrame
df = pd.DataFrame(data)

# Transpose: stations as columns, parameters as rows
df_transposed = df.set_index('Station').T

print(df_transposed)

# Save transposed results
df_transposed.to_csv("/g/data/gb02/mf9078/plots/final/paperFigs/stationGeoscapeParameters.csv")

# Now you can access BUILD_HEIGHT for RE like this:
build_height_re = df_transposed.loc['BUILD_HEIGHT', 'RE']
print(f"\nBUILD_HEIGHT for RE: {build_height_re}")

Station                     RK          RE          LL          CA  \
south_north                160         168         161         165   
west_east                  211         204         181         193   
FRC_URB2D            0.6911965   0.2757736    0.585821   0.5225954   
BUILD_AREA_FRACTION  0.3760219  0.15219732  0.25709438  0.23542662   
BUILD_HEIGHT           8.89515    7.100817   5.0591254   13.035178   
BUILD_SURF_RATIO      0.611469  0.26676702  0.26276067  0.16122687   
HGT                   33.33892   17.560232    23.90398    48.46202   
LU_INDEX                  53.0        56.0        53.0        58.0   

Station                      ED          SS          PT          CN  
south_north                 162         176         176         147  
west_east                   202         168         181         172  
FRC_URB2D            0.38184384  0.43404713   0.5248055   0.5132711  
BUILD_AREA_FRACTION  0.21819936  0.25001618  0.27309507  0.21301246  
BUILD_HEIGHT       